# Stage 2: supervised fine-tuning (QLoRA)

Fine-tunes Qwen2.5-3B-Instruct on the annotated positions from stage 1. The base model loads in 4-bit NF4 through bitsandbytes and only the LoRA adapters train, which fits comfortably on the free T4.

- **Runtime type**: set Runtime > Change runtime type > **T4 GPU** before running.
- **Time**: roughly one to two hours for one epoch.
- **Tracking**: the W&B cell below is optional. Skip it and the run logs offline instead; log in and you get live loss curves.


In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
REPO_URL = "https://github.com/ariyan-tipu/opening-theory-engine.git"

# A plain install rather than -e: editable installs only register when the
# interpreter starts, and a live Colab kernel has already started.
!rm -rf /content/opening-theory-engine
!git clone --depth 1 {REPO_URL} /content/opening-theory-engine
%pip install -q "/content/opening-theory-engine[train,gpu]"
print("install done")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/opening-theory-engine

In [ ]:
# Optional: live loss curves on wandb.ai (free account). This only surfaces the
# key into the environment; the training subprocess below does the actual wandb
# login, so nothing heavy is imported into this kernel.
import os

key = os.environ.get("WANDB_API_KEY")
if not key:
    try:
        from google.colab import userdata

        key = userdata.get("WANDB_API_KEY")
    except Exception:  # noqa: BLE001  (not on Colab, or no secret granted)
        key = None
if key:
    os.environ["WANDB_API_KEY"] = key
    print("W&B key found: this stage will log online.")
else:
    print("No WANDB_API_KEY found: this stage will log offline.")
    print("For live dashboards, add WANDB_API_KEY as a Colab secret and re-run.")

In [ ]:
!ote train sft --config /content/opening-theory-engine/configs/colab-t4.yaml

In [ ]:
print(open("artefacts/sft/sft_metrics.json", encoding="utf-8").read())

The adapter now lives at `artefacts/sft/adapter` on your Drive. Next: `03_align_dpo.ipynb`.
